<a href="https://colab.research.google.com/github/mugalan/intrinsic-rigid-body-control-estimation/blob/main/intrinsic-DEKF/RigidBodyIntinsicEKF_DHSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports and Setup

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy
from sympy import symbols
from sympy import *

import plotly.graph_objects as go
import plotly.express as px
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
from IPython.display import display, Math, Latex

In [ ]:
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims
mr = sims.RigidBodySim()

#Overview

This notebook develops discrete-time Kalman filtering for rigid-body
estimation directly on Lie groups, beginning from the probabilistic
structure of the ordinary Euclidean Kalman filter and progressively
extending it to nonlinear state spaces such as $SO(3)$ and $SE(3)$.

A central viewpoint adopted throughout is that the predicted estimate
$\widetilde g_k^-$ is treated as a *random variable generated by the
stochastic filter model*, in direct analogy with the Euclidean Kalman
prediction. The physical kinematics themselves may therefore remain
deterministic, while process uncertainty is introduced through the
prediction model. This distinction makes the relationship between the
classical Kalman filter and its intrinsic Lie-group counterpart especially
transparent.

For a left-invariant kinematic system with a compatible invariant error,
the deterministic input shared by the true and estimated systems cancels
from the first-order error propagation. In the discrete formulation this
leads to the particularly simple transition
$$
A_{k-1}=I,
$$
while the geometry and the effect of process noise enter through
$$
G_{k-1}
=
-\Delta T\,
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\,\zeta_{k-1})^{-1}.
$$
Thus the local error dynamics recover the familiar discrete linear
Kalman-filter structure, with the Lie-group geometry appearing primarily
through the adjoint action, exponential map, invariant output
linearization, and the corresponding Jacobian.

The notebook develops this construction step by step, including:

- a review of Gaussian conditioning and the discrete linear Kalman filter;

- Lie groups, Lie algebras, adjoint representations, group actions, and
  invariant errors required for intrinsic filtering;

- discrete-time pre-observers and the distinction between left- and
  right-invariant output structures;

- derivation of the intrinsic discrete EKF (DEKF) from the stochastic
  prediction model, including the BCH expansion and the process-noise
  Jacobian;

- comparison between continuous- and discrete-time invariant error
  dynamics and their relationship to classical Kalman filtering;

- specialization to attitude estimation on $SO(3)$ using vector-direction
  measurements;

- an $SO(3)$ IMU sensor-fusion filter with gyroscope-bias estimation,
  together with numerical implementations and examples;

- specialization to rigid-body pose estimation and landmark-aided
  localization on $SE(3)$;

- IMU--GNSS navigation models incorporating attitude, velocity, position,
  gyroscope bias, and accelerometer bias; and

- the use of shifted velocity and position variables
  $$(v_s,o_s)$$
  to absorb the deterministic gravitational acceleration and recover a
  particularly clean invariant error-state structure.

The overall objective is therefore not merely to present an EKF operating
on a Lie group, but to show explicitly how the familiar prediction,
innovation, covariance propagation, and correction steps of the discrete
Kalman filter emerge naturally when estimation error and uncertainty are
expressed intrinsically. This provides a common framework connecting
Euclidean Kalman filtering, invariant observers, Lie-group estimation,
and practical rigid-body sensor-fusion problems.

# The Kalman Filter on $\mathbb{R}^n$

Consider the linear Gaussian process:
\begin{align*}
x_k &= A_{k-1}\,x_{k-1} + G_{k-1}\,w_{k-1}, \\
y_k &= H_k\,x_k + z_k,
\end{align*}
where $w_k \sim \mathscr{N}(0,\Sigma_p)$ and $z_k \sim \mathscr{N}(0,\Sigma_m)$ are mutually independent white noise sequences, also independent of the initial state $x_0 \sim \mathscr{N}(m_0, P_0)$.
Here $y_k$ denotes the random variable representing the measurement at time step $k$, while $y_k^{\mathrm{obs}}$ denotes its observed numerical realization.

---

**Prediction Step (Time Update):**

Define the filter model
\begin{align*}
x^{-}_k &= A_{k-1}\,x^{+}_{k-1} + G_{k-1}\,w_{k-1}, \\
y^{-}_k &= H_k\,x^{-}_k + z_k,
\end{align*}
where
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1}),$$


Because the filter equation is linear and the process noise is Gaussian, the predicted state $x^{-}_k$ is also Gaussian. Taking the expectation of the state equation yields the predicted mean:


$$m_k^- \triangleq \mathbb{E}[x^{-}_k] = A_{k-1}\mathbb{E}[x^{+}_{k-1}] + G_{k-1}\mathbb{E}[w_{k-1}] = A_{k-1}m_{k-1},$$


since $\mathbb{E}[w_{k-1}] = 0$.

The predicted covariance $P_k^-$ is computed by applying the variance operator to the state equation:
\begin{align*}
P_k^- &\triangleq \text{Var}(x^{-}_k) \\
&= A_{k-1}\text{Var}(x^{+}_{k-1})A_{k-1}^T + G_{k-1}\text{Var}(w_{k-1})G_{k-1}^T \\
&= A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T.
\end{align*}
Thus, prior to incorporating the new measurement, our belief of the state is characterized by the prior distribution:


$$x_k^- \sim \mathscr{N}(m_k^-, P_k^-).$$

---

**Measurement Prediction:**

Prior to its physical realization, the upcoming measurement at time step $k$ is treated as the random variable $y_k$. Based on our prior state belief, its expected value is:


$$\mathbb{E}[y^{-}_k] = H_k \mathbb{E}[x_k^-] + \mathbb{E}[z_k] = H_k m_k^-,$$


and its variance is:


$$\text{Var}(y^{-}_k) = H_k \text{Var}(x_k^-) H_k^T + \text{Var}(z_k) = H_k P_k^- H_k^T + \Sigma_m.$$

---

**Joint Prior Distribution:**

The cross-covariance between the predicted state $x_k^-$ and the anticipated measurement random variable $y_k$ is evaluated as:


$$\text{Cov}(x_k^-, y^{-}_k) = \text{Cov}(x_k^-, H_k x_k^- + z_k) = P_k^- H_k^T.$$

Therefore, the joint distribution of the predicted state and the upcoming measurement is a block-structured multivariate Gaussian:
$$\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ H_k m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- H_k^T \\ H_k P_k^- & H_k P_k^- H_k^T + \Sigma_m \end{bmatrix}
\right).$$

---

**Measurement Update (Correction Step):**

At time step $k$, a physical measurement is observed, causing the random variable to take a specific numerical realization: $y_k = y^{\mathrm{obs}}_{k}$.

Applying the standard conditioning properties of multivariate normal distributions, we update our prior belief of the state by slicing the joint distribution at the realization $y^{\mathrm{obs}}_{k}$. This yields the posterior state distribution:


$$x^{+}_k\triangleq (x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) \sim \mathscr{N}(m_k, P_k),$$


where the updated mean $m_k$ and updated covariance $P_k$ are given by:
\begin{align*}
K_k &\triangleq P_k^- H_k^T (H_k P_k^- H_k^T + \Sigma_m)^{-1}, \\
m_k &= m_k^- + K_k (y^{\mathrm{obs}}_{k} - H_k m_k^-), \\
P_k &= (I - K_k H_k) P_k^-.
\end{align*}

Here, $K_k$ is the Kalman Gain, and
$$ m_k=\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}]$$ acts as the updated state estimate, and
$$P_k=\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k})$$ is the new posterior error covariance matrix used to seed the next recursive time-step.

Note that $x^{+}_k$ denotes an abstract random variable distributed according to the conditional law of $x^{-}_k$ given the observed value $y^{\mathrm{obs}}_{k}$.

You may refer to this note on [multivariate Gaussians](https://github.com/mugalan/introduction-to-statistical-learning/blob/main/Multivariate_Gaussian_Distributions.ipynb) for the exact details of extracting the mean and the covariance of the conditional distribution.

---

**Error Dynamics**

For clarity, define the actual estimation error by
$$
\varepsilon_k \triangleq x_k - m_k,
\qquad
\varepsilon_k^- \triangleq x_k - m_k^- .
$$

From the measurement update,
$$
m_k = m_k^- + K_k\left(y_k^{\mathrm{obs}} - H_km_k^-\right),
$$
and using
$$
y_k^{\mathrm{obs}} = H_kx_k + z_k,
$$
we obtain
$$
\begin{aligned}
\varepsilon_k
&= x_k - m_k \\
&= x_k - m_k^- - K_k\left(H_kx_k + z_k - H_km_k^-\right) \\
&= \left(I-K_kH_k\right)(x_k-m_k^-) - K_kz_k \\
&= \left(I-K_kH_k\right)\varepsilon_k^- - K_kz_k .
\end{aligned}
$$

Moreover, from the prediction step,
$$
x_k = A_{k-1}x_{k-1} + G_{k-1}w_{k-1},
\qquad
m_k^- = A_{k-1}m_{k-1},
$$
we have
$$
\varepsilon_k^-
=
A_{k-1}\varepsilon_{k-1}
+
G_{k-1}w_{k-1}.
$$

Therefore,
$$
\boxed{
\varepsilon_k
=
\left(I-K_kH_k\right)A_{k-1}\varepsilon_{k-1}
+
\left(I-K_kH_k\right)G_{k-1}w_{k-1}
-
K_kz_k .
}
$$

## 1-D Example

Consider the scalar linear-Gaussian filter model:
\begin{aligned}
x^-_k &= a\,x^+_{k-1} + w_{k-1},\qquad w_{k-1}\sim\mathscr N(0,\Sigma_q),\\
y^-_k &= h\,x^-_k + z_k,\qquad\;\;\;\; z_k\sim\mathscr N(0,\Sigma_r),
\end{aligned}
where
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1}).$$


**Prediction:**

Let $$x_k^- \sim \mathscr{N}(m_k^-, P_k^-).$$ From the above filter model we have:
\begin{aligned}
m_k^- &= a\,m_{k-1},\\
P_k^- &= a^2 P_{k-1} + \Sigma_q.
\end{aligned}

**Predicted Measurment:**

From the filter model we have that
$$y_k^- \sim \mathscr{N}(hm_k^-, h^2P_k^-+\Sigma_r).$$


Thus
$$
\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ h m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- h \\ P_k^-h & h^2P_k^-+\Sigma_r \end{bmatrix}
\right).
$$



**Measurement Update (Correction Step):**

At time step $k$, a physical measurement is observed, causing the random variable to take a specific numerical realization: $y_k = y^{\mathrm{obs}}_{k}$.

Applying the standard conditioning properties of multivariate normal distributions, we update our prior belief of the state by slicing the joint distribution at the realization $y^{\mathrm{obs}}_{k}$. This yields the posterior state distribution:


$$x^{+}_k\triangleq (x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) \sim \mathscr{N}(m_k, P_k),$$

where the updated mean $m_k$ and updated covariance $P_k$ are given by:
\begin{align*}
K_k &\triangleq \frac{P_k^- h }{(h^2P_k^- + \Sigma_r)}, \\
m_k &= m_k^- + K_k (y^{\mathrm{obs}}_{k} - h m_k^-), \\
P_k &= (1 - K_k h)\,P_k^-
= \Bigl(1 - \frac{P_k^- h^2}{(h^2P_k^- + \Sigma_r)}\Bigr) P_k^- .
\end{align*}

Here, $K_k$ is the Kalman Gain, and
$$ m_k=\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}]$$ acts as the updated state estimate, and
$$P_k=\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k})$$ is the new posterior error covariance matrix used to seed the next recursive time-step.

Note that $x^{+}_k$ denotes an abstract random variable distributed according to the conditional law of $x^{-}_k$ given the observed value $y^{\mathrm{obs}}_{k}$.



In [ ]:
import numpy as np

def make_cv1d(dt: float = 0.1,
              q: float = 1e-2,
              r: float = 1e-1,
              x0=(0.0, 1.0),
              seed: int | None = None) -> sims.LinearGaussianSystemSyms:
    """
    Build a 1D constant-velocity linear Gaussian system:

        x_k = A x_{k-1} + w_{k-1},      w ~ N(0, Q)
        y_k = H x_k       + z_k,        z ~ N(0, R)

    State: x = [position, velocity]^T  (n=2)
    Measurement: y = position (scalar, p=1)

    Parameters
    ----------
    dt : float
        Sampling period Δt.
    q : float
        Continuous white-acceleration noise intensity (process noise scale).
        Discrete-time Q = q * [[dt^3/3, dt^2/2],
                               [dt^2/2, dt     ]].
    r : float
        Measurement noise std. R = [[r^2]] (scalar variance).
    x0 : tuple[float, float]
        Initial state (position, velocity).
    seed : int | None
        Seed for reproducible randomness.

    Returns
    -------
    LinearGaussianSystemSyms
        System with A, H, Sigma_p (Q), Sigma_m (R), and initial state x0.
    """
    A = np.array([[1.0, dt],
                  [0.0, 1.0]], dtype=float)

    # Measure position only (scalar)
    H = np.array([[1.0, 0.0]], dtype=float)  # shape (1,2)

    # Discrete CV process noise covariance (from white-acceleration model)
    Q = q * np.array([[dt**3/3.0, dt**2/2.0],
                      [dt**2/2.0, dt      ]], dtype=float)

    # Measurement noise covariance (scalar)
    R = np.array([[r**2]], dtype=float)

    x0 = np.asarray(x0, dtype=float)
    if x0.shape != (2,):
        raise ValueError(f"`x0` must be shape (2,), got {x0.shape}.")

    rng = np.random.default_rng(seed)
    return sims.LinearGaussianSystemSyms(A=A, H=H, Sigma_p=Q, Sigma_m=R, x0=x0, rng=rng)

# Must have p=1 in your system (scalar measurement). n can be >1.
sys = make_cv1d(dt=0.1, q=5e-2, r=1.0, x0=(0.0, 0.5), seed=7)  # p=1
# Run with simulated Y (T provided)
sys.animate_measurement_gaussians_scalar(T=80, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
                                         frame_ms=120, save_html_path=None, show=True)

# Or if you've already collected Y (shape (T,) or (T,1)):
# _, Y = sys.simulate(T=100)
# sys.animate_measurement_gaussians_scalar(Y=Y, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
#                                          save_html_path="kf_scalar_y_gaussians.html", auto_play=False)

## Simulation 2D-example

We consider a two-dimensional constant-velocity dynamical system. The hidden state at time step $k$ is

$$
x_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)\\
v_x(k)\\
v_y(k)
\end{bmatrix},
$$

where $(p_x(k),p_y(k))$ denote the position components and $(v_x(k),v_y(k))$ denote the velocity components.

The measurement consists only of the two position components:

$$
y_k =
\begin{bmatrix}
p_x^{\mathrm{meas}}(k)\\
p_y^{\mathrm{meas}}(k)
\end{bmatrix}.
$$

The linear Gaussian state-space model is

$$
x_k = A x_{k-1} + G w_{k-1},
$$

$$
y_k = Hx_k + z_k,
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,\Sigma_p),
\qquad
z_k \sim \mathscr{N}(0,\Sigma_m).
$$

The process noise sequence $w_k$, measurement noise sequence $z_k$, and the initial state are assumed mutually independent.

---


Assuming a sampling interval $\Delta t$, the constant-velocity kinematic equations are

$$
p_x(k) = p_x(k-1) + \Delta t\,v_x(k-1),
$$

$$
p_y(k) = p_y(k-1) + \Delta t\,v_y(k-1),
$$

$$
v_x(k) = v_x(k-1),
$$

$$
v_y(k) = v_y(k-1).
$$

Therefore, in matrix form,

$$
x_k =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}
x_{k-1}
+
G w_{k-1}.
$$

Thus,

$$
A =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}.
$$

---

Since the measurement contains only the position components, we have

$$
y_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)
\end{bmatrix}
+
z_k.
$$

Equivalently,

$$
y_k = Hx_k + z_k,
$$

where

$$
H =
\begin{bmatrix}
1 & 0 & 0 & 0\\
0 & 1 & 0 & 0
\end{bmatrix}.
$$

The measurement noise is modeled as

$$
z_k \sim \mathscr{N}(0,\Sigma_m),
$$

with

$$
\Sigma_m = r^2 I_2.
$$

Here, $r$ is the standard deviation of the measurement noise in each position coordinate.

---


A common way to model uncertainty in a constant-velocity system is to assume that the unmodeled acceleration is random. Let

$$
w_{k-1} =
\begin{bmatrix}
a_x(k-1)\\
a_y(k-1)
\end{bmatrix},
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,q^2I_2).
$$

Here, $q$ controls the acceleration-noise intensity.

Over one sampling interval $\Delta t$, the random acceleration affects both position and velocity:

$$
p_x(k)
=
p_x(k-1)
+
\Delta t\,v_x(k-1)
+
\frac{1}{2}\Delta t^2 a_x(k-1),
$$

$$
p_y(k)
=
p_y(k-1)
+
\Delta t\,v_y(k-1)
+
\frac{1}{2}\Delta t^2 a_y(k-1),
$$

$$
v_x(k)
=
v_x(k-1)
+
\Delta t\,a_x(k-1),
$$

$$
v_y(k)
=
v_y(k-1)
+
\Delta t\,a_y(k-1).
$$

Therefore,

$$
G =
\begin{bmatrix}
\frac{1}{2}\Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^2\\
\Delta t & 0\\
0 & \Delta t
\end{bmatrix},
$$

and

$$
\Sigma_p = q I_2.
$$

The induced state-space process covariance is

$$
Q
=
G\Sigma_pG^T.
$$

Since $\Sigma_p=qI_2$, this becomes

$$
Q
=
qGG^T.
$$

Explicitly,

$$
Q
=
q
\begin{bmatrix}
\frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3 & 0\\
0 & \frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3\\
\frac{1}{2}\Delta t^3 & 0 & \Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^3 & 0 & \Delta t^2
\end{bmatrix}.
$$

---

Thus, the full Gaussian filter is

$$
x^-_k =
Ax^+_{k-1}+Gw_{k-1},
\qquad
w_{k-1}\sim\mathscr{N}(0,q^2I_2),
$$

$$
y^-_k =
Hx^-_k+z_k,
\qquad
z_k\sim\mathscr{N}(0,r^2I_2).
$$

In [ ]:
# 2D position-only measurements (p=2), CV model in x & y
def make_cv2d(
    dt=0.1,
    q=1e-4,
    r=0.5,
    x0=(0, 0, 1, 0.5),
    seed=0,
    *,
    use_G=True,
    noise_model="accel_white",
):
    """
    Build a 2D constant-velocity (CV) linear-Gaussian system.

    States: x = [x, y, vx, vy]^T
      A = [[1, 0, dt, 0 ],
           [0, 1, 0 , dt],
           [0, 0, 1 , 0 ],
           [0, 0, 0 , 1 ]]

    Measurements: y = [x, y]^T  (position only)

    Process-noise options
    ---------------------
    - use_G=True, noise_model='accel_white'  (default):
        Uses an explicit G that maps a 2D white acceleration noise (w ~ N(0, q I_2))
        into the state:
            G = [[dt^2/2,     0   ],
                 [   0  ,  dt^2/2],
                 [  dt ,     0   ],
                 [   0 ,    dt   ]]
        Sigma_p = q * I_2   (in w-space)
        → State-space covariance is G Sigma_p G^T
        (This is the common “white-acceleration” CV model.)

    - use_G=False:
        Legacy behavior (no G). We provide the classic state-space Q directly
        corresponding to (velocity random-walk discretization):
            Q_block = [[dt^3/3, dt^2/2],
                       [dt^2/2, dt     ]] * q
        Q = blockdiag(Q_block, Q_block)
        Sigma_p = Q  (already in state space), G=None

    Notes
    -----
    The two variants imply slightly different discrete-time process covariances.
    Pick the one that matches your physical assumption / reference text.
    """
    A = np.array([
        [1, 0, dt,  0],
        [0, 1,  0, dt],
        [0, 0,  1,  0],
        [0, 0,  0,  1],
    ])
    H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ])
    R = np.eye(2) * (r**2)

    if use_G:
        if noise_model != "accel_white":
            raise ValueError("When use_G=True, supported noise_model is only 'accel_white'.")
        # White acceleration injected into vx, vy
        G = np.array([
            [0.5*dt*dt, 0.0       ],
            [0.0      , 0.5*dt*dt],
            [dt       , 0.0       ],
            [0.0      , dt        ],
        ])
        Sigma_p = np.eye(2) * (q**2)  # w-space covariance (2x2)
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Sigma_p, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )
    else:
        # Legacy/state-space Q (velocity random-walk discretization)
        Q_block = np.array([[dt**3/3, dt**2/2],
                            [dt**2/2, dt      ]], dtype=float) * q
        Q = np.block([
            [Q_block,               np.zeros((2, 2))],
            [np.zeros((2, 2)),      Q_block        ],
        ])
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Q, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )

In [ ]:
sys = make_cv2d(dt=0.1, q=1e-2, r=0.5, x0=(0,0, 0.5, -0.2), seed=7)
X2, Y2 = sys.simulate(T=300)
sys.plot_y(Y2, nbins=40, component_labels=["pos_x", "pos_y"])

In [ ]:
# Option A: simulate 300 steps internally, starting from broad prior
M, Yhat= sys.filter_with_kf_and_plot(T=300, Y=Y2, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
                                      component_labels=["pos_x","pos_y"], show=True)

# Option B: if you already have measurements Y, just pass them:
# X, Y = sys.simulate(T=300)
# M, Yhat = sys.filter_with_kf_and_plot(Y=Y, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
#                                       component_labels=["pos_x","pos_y"])

# Intrinsic Extended Kalman Filter on Lie Groups

## Discretized EKF on Lie Groups

We consider a Lie group $G$ with Lie algebra $\mathfrak{g}$ and a left-invariant linear action $\phi:G\times M\to M$ on a vector space $M$ (so each $\phi_g\in \mathrm{GL}(M)$).

Consider the discretized noisy rigid body kinematics
\begin{align*}
g_k &= g_{k-1}\exp{\big(\Delta T\zeta_{k-1}\big)},\\
y_k &= \phi_{g_k^{-1}}(\gamma)+z_k,\qquad p(z_k)=\mathscr{N}(0,\Sigma_m)
\end{align*}
This is a left invariant system on $G$ with a right invariant output.

Define the filter model
\begin{align*}
\widetilde g_k^- &= \widetilde g_{k-1}\exp{\left(\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)},\qquad p(w_{k-1})=\mathscr{N}(0,\Sigma_p)\\
\widetilde g_k &= \exp{\big(L(y_k,\widetilde y_k^-)\big)}\widetilde g_k^-,\\
\widetilde y_k^- &= \phi_{{\widetilde{g}_k^-}^{-1}}(\gamma),
\end{align*}
where $\widetilde g_k^-$ should be interpreted as the predicted
random variable generated by the stochastic filter model (in direct
analogy with the Euclidean prediction), and
$L:M\times M\to \mathscr{G}$ is the innovation term. Define $u_{k-1}\triangleq \exp(\Delta T\zeta_{k-1})$, the a priori error $e_k^-\triangleq g_k(\widetilde g_k^-)^{-1}$, and the a posteriori error $e_k\triangleq g_k\widetilde g_k^{-1}$. Then
\begin{align*}
e_k^- &= g_k(\widetilde{g}_k^-)^{-1} ,\\
e_k &= g_k\widetilde{g}_k^{-1} =
g_k{\widetilde{g}_k^-}^{-1}\exp{\big(-L(y_k,\widetilde y_k^-)\big)}\\
&=
g_{k-1}\exp{\big(\Delta T\zeta_{k-1}\big)}
\exp{\left(-\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)}
\widetilde{g}_{k-1}^{-1}\exp{\big(-L(y_k,\widetilde y_k^-)\big)}\\
&=
g_{k-1}\widetilde{g}_{k-1}^{-1}\mathbb{I}_{\widetilde{g}_{k-1}}
\left(\exp{\big(\Delta T\zeta_{k-1}\big)}
\exp{\left(-\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)}\right)\exp{\big(-L(y_k,\widetilde y_k^-)\big)}\\
&=e_{k-1}\mathbb{I}_{\widetilde{g}_{k-1}}
\Big(\exp{\left(\Delta T\zeta_{k-1}\right)}
\exp{\left(-\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)}\Big)\exp{\big(-L(y_k,\widetilde y_k^-)\big)}
\end{align*}


For the right-invariant output map, define the noise-free measurement
$$
\bar y_k \triangleq \phi_{g_k^{-1}}(\gamma),
\qquad
y_k=\bar y_k+z_k.
$$
Using the invariance property of $L$, the noise-free innovation satisfies
$$
\begin{aligned}
L(\bar y_k,\widetilde y_k^-)
&=
L\left(
\phi_{g_k^{-1}}(\gamma),
\phi_{(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\phi_{g_k}\phi_{g_k^{-1}}(\gamma),
\phi_{g_k}\phi_{(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\gamma,
\phi_{g_k(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\gamma,
\phi_{e_k^-}(\gamma)
\right).
\end{aligned}
$$
For the noisy measurement, since $\phi_{g_k}$ is linear,
$$
\begin{aligned}
L(y_k,\widetilde y_k^-)
&=
L\left(
\phi_{g_k}(y_k),
\phi_{g_k}(\widetilde y_k^-)
\right)\\
&=
L\left(
\gamma+\phi_{g_k}(z_k),
\phi_{e_k^-}(\gamma)
\right).
\end{aligned}
$$


---

Define $e_k=e^{\eta_k}$ and recall
\begin{align*}
\log(\exp X\,\exp Y)
&=
X+\Phi(X)Y+O(\|Y\|^2),
\end{align*}
where
\begin{align*}
\Phi(\eta)
&=
\frac{\operatorname{ad}_\eta e^{\operatorname{ad}_\eta}}
{e^{\operatorname{ad}_\eta}-I} \\
&=
I+\sum_{m=1}^{\infty}
\frac{(-1)^{m+1}}{m(m+1)}
\left(e^{\operatorname{ad}_\eta}-I\right)^m .
\end{align*}

To expand the process-noise contribution, let
\begin{align*}
A_{k-1}
&\triangleq
\Delta T\zeta_{k-1},
&
\delta_{k-1}
&\triangleq
\Delta T w_{k-1}.
\end{align*}
Expanding about $\delta_{k-1}=0$ gives
\begin{align*}
\exp(A_{k-1})
\exp\left(-(A_{k-1}+\delta_{k-1})\right)
&=
\exp\left(
-J_l(A_{k-1})\delta_{k-1}
+O(\|\delta_{k-1}\|^2)
\right),
\end{align*}
where
\begin{align*}
J_l(A)
&=
\frac{e^{\operatorname{ad}_A}-I}
{\operatorname{ad}_A}
=
\Phi(-A)^{-1}.
\end{align*}
Therefore,
\begin{align*}
&\exp\left(\Delta T\zeta_{k-1}\right)
\exp\left(-\Delta T(\zeta_{k-1}+w_{k-1})\right)\\
&\qquad=
\exp\left(
-\Delta T
\Phi(-\Delta T\zeta_{k-1})^{-1}w_{k-1}
+
O\left((\Delta T)^2\|w_{k-1}\|^2\right)
\right).
\end{align*}

Define
\begin{align*}
Y
&\triangleq
-\Delta T
\Phi(-\Delta T\zeta_{k-1})^{-1}w_{k-1}
+
O\left((\Delta T)^2\|w_{k-1}\|^2\right)
=
-\Delta T w_{k-1}
+O((\Delta T)^2).
\end{align*}



---

Then we have
\begin{align*}
\exp(\eta_k)
&=
\exp(\eta_{k-1})
\mathbb{I}_{\widetilde g_{k-1}}
\exp(Y)
\exp\big(-L(y_k,\widetilde y_k^-)\big)\\
&=
\exp(\eta_{k-1})
\exp\left(\operatorname{Ad}_{\widetilde g_{k-1}}Y\right)
\exp\big(-L(y_k,\widetilde y_k^-)\big).
\end{align*}

Applying
$$
\log(\exp X\exp Y)
=
X+\Phi(X)Y+O(\|Y\|^2)
$$
to the last two exponential factors gives
\begin{align*}
\exp(\eta_k)
&=
\exp(\eta_{k-1})
\exp\Big(
\operatorname{Ad}_{\widetilde g_{k-1}}Y
-
\Phi\left(\operatorname{Ad}_{\widetilde g_{k-1}}Y\right)
L(y_k,\widetilde y_k^-)
+
O(\|L(y_k,\widetilde y_k^-)\|^2)
\Big).
\end{align*}

Applying the same expansion once more yields
\begin{align*}
\eta_k
&=
\eta_{k-1}
+
\Phi(\eta_{k-1})
\Big(
\operatorname{Ad}_{\widetilde g_{k-1}}Y
-
\Phi\left(\operatorname{Ad}_{\widetilde g_{k-1}}Y\right)
L(y_k,\widetilde y_k^-)
\Big)
+
H.O.T.
\end{align*}

Recalling
\begin{align*}
\Phi(\eta)
&=
I+\frac{1}{2}\operatorname{ad}_{\eta}
+\frac{1}{12}\operatorname{ad}_{\eta}^2
+O(\|\eta\|^3),
\end{align*}
and retaining terms that are first order in the estimation error,
process noise, and measurement innovation, while neglecting their
quadratic and cross products, we obtain
\begin{align*}
\eta_k
&=
\eta_{k-1}
+
\operatorname{Ad}_{\widetilde g_{k-1}}Y
-
L(y_k,\widetilde y_k^-)
+
H.O.T.
\end{align*}

Using
\begin{align*}
Y
&=
-\Delta T
\Phi(-\Delta T\zeta_{k-1})^{-1}w_{k-1}
+
O\left((\Delta T)^2\|w_{k-1}\|^2\right),
\end{align*}
gives
\begin{align*}
\eta_k
&=
\eta_{k-1}
-\Delta T
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})^{-1}w_{k-1}
-
L(y_k,\widetilde y_k^-)
+
H.O.T.
\end{align*}

Thus, defining
\begin{align*}
G_{k-1}
&\triangleq
-\Delta T
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})^{-1},
\end{align*}
the first-order intrinsic error dynamics take the particularly simple form
\begin{align*}
\eta_k
&=
\eta_{k-1}
+
G_{k-1}w_{k-1}
-
L(y_k,\widetilde y_k^-)
+
H.O.T.
\end{align*}

Define the a priori linearized error by
\begin{align*}
\eta_k^-
&=
\eta_{k-1}+G_{k-1}w_{k-1},
\end{align*}
where
\begin{align*}
G_{k-1}
&\triangleq
-\Delta T\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})^{-1}.
\end{align*}

Let $\ell:M\times M\to\mathbb{R}^m$ denote the invariant measurement
residual, with $\ell(\gamma,\gamma)=0$, and define the innovation by
\begin{align*}
L(y_k,\widetilde y_k^-)
&=
K_k\ell(y_k,\widetilde y_k^-).
\end{align*}
Assume that the residual has the first-order linearization
\begin{align*}
\ell(y_k,\widetilde y_k^-)
&=
H_k\eta_k^-+z_k,
\end{align*}
where $z_k$ denotes the measurement noise expressed in the corresponding
invariant residual coordinates. Hence,
\begin{align*}
L(y_k,\widetilde y_k^-)
&=
K_k\left(
H_k\eta_k^-+z_k
\right)\\
&=
K_k\left(
H_k\left(\eta_{k-1}+G_{k-1}w_{k-1}\right)+z_k
\right).
\end{align*}

For the noise-free residual, define
\begin{align*}
\Gamma_\phi\xi
&\triangleq
\left.
\frac{d}{d\varepsilon}
\phi_{\exp(\varepsilon\xi)}(\gamma)
\right|_{\varepsilon=0},
\\
J_\ell
&\triangleq
\left.D_2\ell(\gamma,\cdot)\right|_{\cdot=\gamma}.
\end{align*}
Then the measurement Jacobian is
\begin{align*}
H_k
&=
J_\ell\Gamma_\phi.
\end{align*}

Using the first-order intrinsic error update
\begin{align*}
\eta_k
&=
\eta_k^-
-
L(y_k,\widetilde y_k^-),
\end{align*}
we obtain
\begin{align*}
\eta_k
&=
\eta_{k-1}
+
G_{k-1}w_{k-1}
-
K_kH_k\eta_{k-1}
-
K_kH_kG_{k-1}w_{k-1}
-
K_kz_k\\
&=
(I-K_kH_k)\eta_{k-1}
+
(I-K_kH_k)G_{k-1}w_{k-1}
-
K_kz_k.
\end{align*}

Thus, to first order, the intrinsic error dynamics have exactly the
same algebraic form as those of the discrete linear Kalman filter, with
\begin{align*}
A_{k-1}=I.
\end{align*}

### The DEKF Equations



Prediction (state on the group and covariance):
\begin{align*}
\widetilde g_k^-
&=
\widetilde g_{k-1}
\exp\left(\Delta T\zeta_{k-1}\right),\\
P_k^-
&=
A_{k-1}P_{k-1}A_{k-1}^{\top}
+
G_{k-1}\Sigma_pG_{k-1}^{\top},
\end{align*}
where
\begin{align*}
A_{k-1}
&=
I,\\
G_{k-1}
&=
-\Delta T
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})^{-1}.
\end{align*}
Thus,
\begin{align*}
P_k^-
&=
P_{k-1}
+
G_{k-1}\Sigma_pG_{k-1}^{\top}.
\end{align*}

The predicted output is
\begin{align*}
\widetilde y_k^-
&=
\phi_{(\widetilde g_k^-)^{-1}}(\gamma).
\end{align*}

Define the invariant measurement residual
\begin{align*}
r_k
&\triangleq
\ell(y_k,\widetilde y_k^-),
\end{align*}
whose first-order model is
\begin{align*}
r_k
&=
H_k\eta_k^-+z_k,
\end{align*}
where $\Sigma_m$ denotes the covariance of $z_k$ in the corresponding
linearized residual coordinates.

Correction (gain, group update, and covariance update):
\begin{align*}
K_k
&=
P_k^-H_k^{\top}
\left(
H_kP_k^-H_k^{\top}+\Sigma_m
\right)^{-1},\\
L(y_k,\widetilde y_k^-)
&=
K_kr_k,\\
\widetilde g_k
&=
\exp\left(
L(y_k,\widetilde y_k^-)
\right)\widetilde g_k^-,\\
P_k
&=
(I-K_kH_k)P_k^-.
\end{align*}

Thus, to first order, the intrinsic estimation error and its covariance
obey the same algebraic recursion as the discrete linear Kalman filter,
with
\begin{align*}
A_{k-1}=I.
\end{align*}

## Application to Rigid Body Motion Estimation

### Rigidbody Kinematics

Rigid-body kinematics on $SE(3)$ are given by
\begin{align*}
\dot g &= g\zeta,
\end{align*}
where $g\in SE(3)$ and $\zeta\in\mathfrak{se}(3)$. Writing out the blocks,
\begin{align*}
g &=
\begin{bmatrix}
R & o\\
0 & 1
\end{bmatrix},
\qquad
\zeta =
\begin{bmatrix}
\widehat{\Omega} & V\\
0 & 0
\end{bmatrix},
\end{align*}
with $R\in SO(3)$, $o,V\in\mathbb{R}^3$, and
$\widehat{\Omega}\in\mathfrak{so}(3)$. Here $R$ maps body-frame
coordinates to the fixed spatial frame, $o$ is the position of the
body-frame origin expressed in the spatial frame, $\Omega$ is the
body-frame angular velocity, and $V$ is the body-frame translational
velocity. In particular,
\begin{align*}
\dot R &= R\widehat{\Omega},
&
\dot o &= RV.
\end{align*}

For $\zeta=(\Omega,V)$ and $\xi=(\Phi,U)$ in $\mathfrak{se}(3)$, the
Lie-algebra adjoint action is
\begin{align*}
\operatorname{ad}_{\zeta}\xi
&=
(\Omega\times\Phi,\,
\Omega\times U-\Phi\times V),
\end{align*}
so that, with the coordinate ordering $(\Omega,V)$,
\begin{align*}
\operatorname{ad}_{\zeta}
&=
\begin{bmatrix}
\widehat{\Omega} & 0\\
\widehat{V} & \widehat{\Omega}
\end{bmatrix}.
\end{align*}

Consider the left action $\phi:SE(3)\times\mathbb{R}^4\to\mathbb{R}^4$
given by homogeneous multiplication:
\begin{align*}
\phi_g
\begin{pmatrix}
x\\
\alpha
\end{pmatrix}
&=
\begin{bmatrix}
R & o\\
0 & 1
\end{bmatrix}
\begin{bmatrix}
x\\
\alpha
\end{bmatrix},
\qquad
x\in\mathbb{R}^3,\quad \alpha\in\mathbb{R}.
\end{align*}

A point with fixed spatial coordinates $x\in\mathbb{R}^3$ is represented
in homogeneous coordinates by
$$
\gamma=
\begin{bmatrix}
x\\
1
\end{bmatrix}.
$$
For $\zeta=(\Omega,V)$, the infinitesimal action is
\begin{align*}
(T_e\phi\circ\zeta)(\gamma)
&=
\begin{bmatrix}
\widehat{\Omega} & V\\
0 & 0
\end{bmatrix}
\begin{bmatrix}
x\\
1
\end{bmatrix}.
\end{align*}

A key term appearing in the intrinsic linearization is
\begin{align*}
\phi_{\widetilde g^{-1}}
\left(
(T_e\phi\circ\operatorname{Ad}_{\widetilde g})\zeta
\right)(\gamma)
&=
\zeta\widetilde g^{-1}\gamma\\
&=
\begin{bmatrix}
\widehat{\Omega}\widetilde R^\top(x-\widetilde o)+V\\
0
\end{bmatrix},
\end{align*}
where $\widetilde g=(\widetilde R,\widetilde o)$. Therefore, restricting
to the first three components,
\begin{align*}
\widehat{\Omega}\widetilde R^\top(x-\widetilde o)+V
&=
\begin{bmatrix}
-\widetilde R^\top\widehat{(x-\widetilde o)}\widetilde R
&
I_{3\times3}
\end{bmatrix}
\begin{bmatrix}
\Omega\\
V
\end{bmatrix}.
\end{align*}
This gives the explicit structure required for the measurement
linearization $H_k$ in the SLAM example below.

### Attitude Estimation from IMUs

#### Without Gyro Bias

In this case take $G=SO(3)$, $M=\mathfrak{so}(3)\simeq\mathbb{R}^3$,
and let the action be the adjoint action $\phi=\operatorname{Ad}$, so that
\begin{align*}
\operatorname{Ad}_R x
&=
Rx,
\qquad
R\in SO(3),\quad x\in\mathbb{R}^3.
\end{align*}
For direction measurements we restrict $x$ to
$\mathbb{S}^2\subset\mathbb{R}^3$.

The Lie bracket is
\begin{align*}
\operatorname{ad}_{\Omega}\Phi
&=
\widehat{\Omega}\Phi
=
\Omega\times\Phi.
\end{align*}

Suppose that two known inertial directions
$e_1,e_2\in\mathbb{S}^2$ are measured in the body frame:
\begin{align*}
y_k
&=
\begin{bmatrix}
R_k^\top e_1\\
R_k^\top e_2
\end{bmatrix}
+
z_k.
\end{align*}

For the adjoint action, the infinitesimal action satisfies
\begin{align*}
(T_e\phi\circ\Omega)x
&=
\widehat{\Omega}x.
\end{align*}
Furthermore,
\begin{align*}
\left(
\phi_{R^\top}
\circ
T_e\phi
\circ
\operatorname{Ad}_R\cdot\Omega
\right)x
&=
-\left(
R^\top\widehat{x}R
\right)\Omega.
\end{align*}

For consistency with the right-invariant estimation error
\begin{align*}
e_k^-
&=
R_k(\widetilde R_k^-)^\top
=
\exp(\widehat{\eta_k^-}),
\end{align*}
define the invariant measurement residual
\begin{align*}
r_k
&\triangleq
\begin{bmatrix}
\widetilde R_k^-
\left(
y_{1,k}-(\widetilde R_k^-)^\top e_1
\right)\\
\widetilde R_k^-
\left(
y_{2,k}-(\widetilde R_k^-)^\top e_2
\right)
\end{bmatrix}.
\end{align*}
Since
\begin{align*}
\widetilde R_k^-R_k^\top
&=
(e_k^-)^{-1}
=
\exp(-\widehat{\eta_k^-}),
\end{align*}
we obtain, to first order,
\begin{align*}
r_k
&=
H_k\eta_k^-+\bar z_k+O(\|\eta_k^-\|^2),
\end{align*}
where
\begin{align*}
H_k
&=
\begin{bmatrix}
\widehat{e_1}\\
\widehat{e_2}
\end{bmatrix},
\end{align*}
and
\begin{align*}
\bar z_k
&=
\begin{bmatrix}
\widetilde R_k^- & 0\\
0 & \widetilde R_k^-
\end{bmatrix}z_k.
\end{align*}
For isotropic vector-measurement noise, the rotation leaves the
measurement covariance unchanged, so that
\begin{align*}
\operatorname{Cov}(\bar z_k)
&=
\Sigma_m.
\end{align*}

The discrete intrinsic EKF on $SO(3)$ is therefore

Prediction:
\begin{align*}
\widetilde R_k^-
&=
\widetilde R_{k-1}
\exp\left(
\Delta T\,\widehat{\Omega}_{k-1}
\right),\\
G_{k-1}
&=
-\Delta T\,
\widetilde R_{k-1}
\Phi(-\Delta T\Omega_{k-1})^{-1},\\
P_k^-
&=
P_{k-1}
+
G_{k-1}\Sigma_qG_{k-1}^\top.
\end{align*}

The predicted output is
\begin{align*}
\widetilde y_k^-
&=
\begin{bmatrix}
(\widetilde R_k^-)^\top e_1\\
(\widetilde R_k^-)^\top e_2
\end{bmatrix}.
\end{align*}

Correction:
\begin{align*}
K_k
&=
P_k^-H_k^\top
\left(
H_kP_k^-H_k^\top+\Sigma_m
\right)^{-1},\\
\delta_k
&=
K_kr_k,\\
\widetilde R_k
&=
\exp(\widehat{\delta_k})\widetilde R_k^-,\\
P_k
&=
(I-K_kH_k)P_k^-.
\end{align*}

Since
\begin{align*}
\Phi(-\Delta T\Omega_{k-1})^{-1}
&=
I_{3\times3}+O(\Delta T),
\end{align*}
the first-order approximation of the process-noise map is
\begin{align*}
G_{k-1}
&\approx
-\Delta T\,\widetilde R_{k-1}.
\end{align*}

These equations are the attitude-only specialization of the intrinsic
discrete EKF to $SO(3)$ with two vector measurements.

##### Simulation Example

In [ ]:
import numpy as np


def generate_noisy_so3_direction_data(
    T=500,
    dt=0.01,
    omega_body=(0.4, 0.2, 0.1),
    inertial_directions=None,
    sigma_gyro=0.01,
    sigma_y=0.03,
    R0=None,
    seed=7,
):
    """
    Generate synthetic SO(3) attitude data with noisy gyro inputs and
    additive Gaussian vector-direction measurements.

    True dynamics:
        R_k = R_{k-1} exp(dt * hat(Omega_{k-1}))

    Gyro measurement:
        Omega_data[k-1] = Omega_{k-1} + w_{k-1}
        w_{k-1} ~ N(0, sigma_gyro^2 I_3)

    Direction measurement:
        y_k = [R_k^T e_1; R_k^T e_2; ...] + z_k
        z_k ~ N(0, sigma_y^2 I_{3m})

    Returns
    -------
    R_true : (T, 3, 3)
        True attitudes R_k, k=1,...,T.
    Omega_data : (T, 3)
        Noisy gyro samples used for the corresponding propagation step.
    Y_data : (T, 3m)
        Noisy stacked direction measurements.
    Y_clean : (T, 3m)
        Noise-free stacked direction measurements.
    """

    rng = np.random.default_rng(seed)

    if inertial_directions is None:
        inertial_directions = np.array([
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
        ])

    E = np.asarray(inertial_directions, dtype=float)

    if E.ndim != 2 or E.shape[1] != 3:
        raise ValueError("inertial_directions must have shape (m, 3).")

    norms = np.linalg.norm(E, axis=1)
    if np.any(norms < 1e-12):
        raise ValueError("Inertial directions must be nonzero.")

    # Ensure e_i ∈ S^2.
    E = E / norms[:, None]
    m = E.shape[0]

    if R0 is None:
        R = np.eye(3)
    else:
        R = np.asarray(R0, dtype=float).reshape(3, 3)

    omega_body = np.asarray(omega_body, dtype=float).reshape(3)

    def hat(x):
        x = np.asarray(x, dtype=float).reshape(3)
        return np.array([
            [0.0, -x[2],  x[1]],
            [x[2],  0.0, -x[0]],
            [-x[1], x[0], 0.0],
        ])

    def exp_so3(omega):
        omega = np.asarray(omega, dtype=float).reshape(3)
        theta = np.linalg.norm(omega)
        W = hat(omega)

        if theta < 1e-12:
            return np.eye(3) + W + 0.5 * W @ W

        A = np.sin(theta) / theta
        B = (1.0 - np.cos(theta)) / theta**2

        return np.eye(3) + A * W + B * W @ W

    R_true = np.empty((T, 3, 3))
    Omega_data = np.empty((T, 3))
    Y_clean = np.empty((T, 3 * m))
    Y_data = np.empty((T, 3 * m))

    for k in range(T):

        # Gyro sample corresponding to the propagation R_{k-1} -> R_k
        Omega_data[k] = (
            omega_body
            + rng.normal(0.0, sigma_gyro, size=3)
        )

        # True attitude propagated using the noise-free angular velocity
        R = R @ exp_so3(dt * omega_body)
        R_true[k] = R

        # Noise-free direction measurement at R_k
        y_clean = np.concatenate([
            R.T @ e for e in E
        ])

        # Additive Gaussian measurement noise
        z_k = rng.normal(
            0.0,
            sigma_y,
            size=3 * m,
        )

        Y_clean[k] = y_clean
        Y_data[k] = y_clean + z_k

    return R_true, Omega_data, Y_data, Y_clean

In [ ]:
E = np.array([
    #[1.0, 0.0, 0.0],
    #[0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])

R_true, Omega_data, Y_data, Y_clean = generate_noisy_so3_direction_data(
    T=500,
    dt=0.01,
    omega_body=np.array([0.4, 0.2, 0.1]),
    inertial_directions=E,
    sigma_gyro=0.01,
    sigma_y=0.03,
    seed=7,
)

In [ ]:
kf = sims.LinearKF(use_joseph=True, symmetrize=True)
iekf = sims.SO3IMUSensorFusionEKF(kf=kf, inertial_directions=E, dt=0.01)

results = iekf.run_filter(
    R0_plus=np.eye(3),
    P0=1e-2 * np.eye(3),
    Omegas=Omega_data,
    Ys=Y_data,
    Sigma_q=1e-4 * np.eye(3),
    Sigma_m=1e-2 * np.eye(3),
)

fig = iekf.plot_measurements(Y_data, results["yhat"])
fig.show()

#### With Gyro Bias

The augmented local error state is
$$
\eta_k
=
\begin{bmatrix}
\eta_{R,k}\\
\eta_{b,k}
\end{bmatrix}
\in\mathbb{R}^6,
$$
where the right-invariant attitude error is defined by
\begin{align*}
R_k\widetilde R_k^\top
&=
\exp(\widehat{\eta_{R,k}}),
\end{align*}
and the gyro-bias error is
\begin{align*}
\eta_{b,k}
&\triangleq
\Omega_{b,k}-\widetilde\Omega_{b,k}.
\end{align*}

Define the bias-corrected angular-velocity estimate
\begin{align*}
\widetilde\omega_{k-1}
&\triangleq
\Omega_{k-1}-\widetilde\Omega_{b,k-1}.
\end{align*}

The prediction equations are
\begin{align*}
\widetilde R_k^-
&=
\widetilde R_{k-1}
\exp\left(
\Delta T\,\widehat{\widetilde\omega}_{k-1}
\right),\\
\widetilde\Omega_{b,k}^-
&=
\widetilde\Omega_{b,k-1}.
\end{align*}

For vector measurements $e_1,e_2\in\mathbb{S}^2$, the predicted output is
\begin{align*}
\widetilde y_k^-
&=
\begin{bmatrix}
(\widetilde R_k^-)^\top e_1\\
(\widetilde R_k^-)^\top e_2
\end{bmatrix}.
\end{align*}

Define the invariant measurement residual
\begin{align*}
r_k
&\triangleq
\begin{bmatrix}
\widetilde R_k^-
\left(
y_{1,k}-(\widetilde R_k^-)^\top e_1
\right)\\
\widetilde R_k^-
\left(
y_{2,k}-(\widetilde R_k^-)^\top e_2
\right)
\end{bmatrix}.
\end{align*}

Its first-order linearization is
\begin{align*}
r_k
&=
H_k\eta_k^-+\bar z_k,
\end{align*}
with
\begin{align*}
H_k
&=
\begin{bmatrix}
\widehat e_1 & 0_{3\times3}\\
\widehat e_2 & 0_{3\times3}
\end{bmatrix}.
\end{align*}

Define
\begin{align*}
F_{k-1}
&\triangleq
-\Delta T\,
\widetilde R_{k-1}
\Phi(-\Delta T\widetilde\omega_{k-1})^{-1}.
\end{align*}

Then the augmented linearized prediction matrices are
\begin{align*}
A_{k-1}
&=
\begin{bmatrix}
I_{3\times3} & F_{k-1}\\
0 & I_{3\times3}
\end{bmatrix},\\
G_{k-1}
&=
\begin{bmatrix}
F_{k-1} & 0\\
0 & I_{3\times3}
\end{bmatrix}.
\end{align*}

Thus,
\begin{align*}
P_k^-
&=
A_{k-1}P_{k-1}A_{k-1}^\top
+
G_{k-1}\Sigma_qG_{k-1}^\top,\\
K_k
&=
P_k^-H_k^\top
\left(
H_kP_k^-H_k^\top+\Sigma_m
\right)^{-1},\\
P_k
&=
(I-K_kH_k)P_k^-.
\end{align*}

The correction is
$$
\begin{bmatrix}
\delta_{R,k}\\
\delta_{b,k}
\end{bmatrix}
=
K_kr_k,
$$
followed by
\begin{align*}
\widetilde R_k
&=
\exp(\widehat{\delta_{R,k}})
\widetilde R_k^-,\\
\widetilde\Omega_{b,k}
&=
\widetilde\Omega_{b,k}^-+\delta_{b,k}.
\end{align*}

For sufficiently small $\Delta T$,
\begin{align*}
F_{k-1}
&=
-\Delta T\,\widetilde R_{k-1}
+O((\Delta T)^2),
\end{align*}
so that the first-order approximation is
\begin{align*}
A_{k-1}
&\approx
\begin{bmatrix}
I_{3\times3}
&
-\Delta T\,\widetilde R_{k-1}\\
0&I_{3\times3}
\end{bmatrix},\\
G_{k-1}
&\approx
\begin{bmatrix}
-\Delta T\,\widetilde R_{k-1}
&
0\\
0&I_{3\times3}
\end{bmatrix}.
\end{align*}

In [ ]:
Y_data

In [ ]:
Omega_data

##### Example with real data

[The Zurich Urban Micro Aerial Vehicle Dataset](https://www.kaggle.com/datasets/mrisdal/zurich-urban-micro-aerial-vehicle)

In [ ]:
!pip install "git+https://github.com/mugalan/data-analysis-tool.git"

In [ ]:
from data_analysis import DataInspector, PlottingMethods
inspector = DataInspector()

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrisdal/zurich-urban-micro-aerial-vehicle")

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df_gyro=pd.read_csv(path+"/RawGyro.csv")
df_accel=pd.read_csv(path+"/RawAccel.csv")

In [ ]:
df_merged = pd.merge(df_accel, df_gyro, on='Timpstemp', how='inner', suffixes=('_accel', '_gyro'))
display(df_merged.head())

In [ ]:
inspector.df=df_merged

In [ ]:
column_names=[' x_accel', ' y_accel', ' z_accel', ' x_gyro', ' y_gyro', ' z_gyro']
inspector.plot_numerical(column_names=column_names)

In [ ]:
Omega_data=df_gyro[[' x', ' y', ' z']].to_numpy()
raw_acc=df_accel[[' x', ' y', ' z']].to_numpy()

In [ ]:
Y_data = raw_acc / np.linalg.norm(raw_acc, axis=1, keepdims=True)

In [ ]:
Y_data

In [ ]:
def lowpass_accel(acc, alpha=0.02):
    acc_lp = np.zeros_like(acc)
    acc_lp[0] = acc[0]

    for k in range(1, len(acc)):
        acc_lp[k] = (1 - alpha) * acc_lp[k-1] + alpha * acc[k]

    return acc_lp

In [ ]:
low_pass_acc=lowpass_accel(raw_acc)

In [ ]:
Y_data = low_pass_acc/ np.linalg.norm(low_pass_acc, axis=1, keepdims=True)

In [ ]:
E = np.array([
    [0.0, 0.0, -1.0],   # gravity direction for your accelerometer convention
])

kf = sims.LinearKF(use_joseph=True, symmetrize=True)

iekf = sims.SO3IMUSensorFusionBiasEKF(
    kf=kf,
    inertial_directions=E,
    dt=0.01,
)

#Sigma_gyro = 0.02**2 * np.eye(3)
Sigma_gyro = 0.09**2 * np.eye(3)
#Sigma_bias = 1e-5**2 * np.eye(3)
Sigma_bias = 4e-4**2 * np.eye(3)

Sigma_q = np.block([
    [Sigma_gyro, np.zeros((3, 3))],
    [np.zeros((3, 3)), Sigma_bias],
])

#Sigma_m = 0.18**2 * np.eye(3)
Sigma_m = 0.02**2 * np.eye(3)


#############
results = iekf.run_filter(
    R0_plus=np.eye(3),
    b0_plus=np.zeros(3),
    P0=np.diag([1e-2, 1e-2, 1e-2, 1e-4, 1e-4, 1e-4]),
    Omegas=Omega_data,
    Ys=Y_data,
    Sigma_q=Sigma_q,
    Sigma_m=Sigma_m,
)

fig = iekf.plot_measurements(Y_data, results["yhat"])
fig.show()

fig_b = iekf.plot_bias(results)
fig_b.show()

**Note that:** With only a single gravity-direction measurement, the attitude is observable only up to rotations about the gravity axis; consequently, absolute yaw is unobservable. A second non-collinear reference direction, such as a magnetometer measurement, is required to make the full attitude observable.

### Landmark-Aided Pose Estimation

Let $e$ be a fixed world frame and let $p_i$, $i=1,\dots,N$, denote
fixed landmarks with known spatial coordinates $x_i\in\mathbb{R}^3$.
Write
$$
\gamma_i=
\begin{bmatrix}
x_i\\
1
\end{bmatrix}
\in\mathbb{R}^4
$$
for their homogeneous coordinates. The rigid-body kinematics are
\begin{align*}
\dot g
&=
g\zeta,
\qquad
g\in SE(3),
\end{align*}
and the corresponding body-frame coordinates of landmark $i$ are
\begin{align*}
X_i
&=
R^\top(x_i-o).
\end{align*}
Equivalently, in homogeneous coordinates,
\begin{align*}
\phi_{g^{-1}}(\gamma_i)
&=
\begin{bmatrix}
X_i\\
1
\end{bmatrix}.
\end{align*}

The measured landmark coordinates are modeled as
\begin{align*}
y_i
&=
X_i+z_i,
\qquad
z_i\sim\mathscr{N}(0,\Sigma_{m,i}).
\end{align*}
Process uncertainty is introduced through the stochastic prediction
model of the intrinsic DEKF as described previously.

Define the right-invariant a priori pose error by
\begin{align*}
e_k^-
&=
g_k(\widetilde g_k^-)^{-1}
=
\exp(\eta_k^-),
\qquad
\eta_k^-\in\mathfrak{se}(3).
\end{align*}

For each landmark, define the invariant residual
\begin{align*}
r_{i,k}
&\triangleq
\widetilde R_k^-
\left(
y_{i,k}-\widetilde y_{i,k}^-
\right),
\end{align*}
where
\begin{align*}
\widetilde y_{i,k}^-
&=
(\widetilde R_k^-)^\top
(x_i-\widetilde o_k^-).
\end{align*}

Writing
$$
\eta_k^-=
\begin{bmatrix}
\eta_{R,k}^-\\
\eta_{o,k}^-
\end{bmatrix},
$$
the first-order invariant residual is
\begin{align*}
r_{i,k}
&=
\begin{bmatrix}
\widehat{x_i} & -I_{3\times3}
\end{bmatrix}
\eta_k^-
+
\bar z_{i,k}
+
O(\|\eta_k^-\|^2),
\end{align*}
where
\begin{align*}
\bar z_{i,k}
&=
\widetilde R_k^-z_{i,k}.
\end{align*}

Stacking all landmark residuals gives
\begin{align*}
r_k
&=
H\eta_k^-+\bar z_k+O(\|\eta_k^-\|^2),
\end{align*}
with
\begin{align*}
H
&=
\begin{bmatrix}
\widehat{x_1} & -I_{3\times3}\\
\widehat{x_2} & -I_{3\times3}\\
\vdots & \vdots\\
\widehat{x_N} & -I_{3\times3}
\end{bmatrix}.
\end{align*}

The intrinsic discrete prediction matrices are
\begin{align*}
A_{k-1}
&=
I_{6\times6},\\
G_{k-1}
&=
-\Delta T\,
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})^{-1},
\end{align*}
where
$$
\operatorname{Ad}_{\widetilde g_{k-1}}
=
\begin{bmatrix}
\widetilde R_{k-1} & 0\\
\widehat{\widetilde o}_{k-1}\widetilde R_{k-1}
&
\widetilde R_{k-1}
\end{bmatrix},
$$
and
$$
\operatorname{ad}_{\zeta_{k-1}}
=
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat V_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}.
$$

For sufficiently small $\Delta T$,
\begin{align*}
G_{k-1}
&\approx
-\Delta T
\begin{bmatrix}
\widetilde R_{k-1} & 0\\
\widehat{\widetilde o}_{k-1}\widetilde R_{k-1}
&
\widetilde R_{k-1}
\end{bmatrix}.
\end{align*}

The intrinsic discrete EKF prediction is
\begin{align*}
\widetilde g_k^-
&=
\widetilde g_{k-1}
\exp(\Delta T\zeta_{k-1}),\\
P_k^-
&=
P_{k-1}
+
G_{k-1}\Sigma_pG_{k-1}^\top.
\end{align*}

The correction is
\begin{align*}
K_k
&=
P_k^-H^\top
\left(
HP_k^-H^\top+\Sigma_m
\right)^{-1},\\
\delta_k
&=
K_kr_k,\\
\widetilde g_k
&=
\exp(\delta_k)\widetilde g_k^-,\\
P_k
&=
(I-K_kH)P_k^-.
\end{align*}

### The IMU+GNSS sensor fusion problem

#### Without Bias

A representative application is rigid-body orientation/pose estimation with IMU and GNSS. Gyroscopes measure body-frame angular velocity $\Omega$, accelerometers measure $A_m$ (specific force), and GNSS provides $o,\dot o$. A convenient reformulation introduces
\begin{align*}
o_s(t)&\triangleq o(t)+\tfrac{1}{2}gt^2 e_3,\qquad\\
v_s(t)&\triangleq \dot o_s(t)=\dot o(t)+g t\,e_3,\qquad\\
R\,A_m&=\ddot o_s(t)=\ddot o(t)+g e_3,
\end{align*}
giving the continuous-time model
\begin{align*}
\dot R &= R\,\widehat\Omega, \\
\dot v_s &= R\,A_m, \\
\dot o_s &= v_s, \\
y_o &= o_s, \\
y_v &= v_s.
\end{align*}
Defining the homogeneous state
\begin{align*}
X &\triangleq\;
\begin{bmatrix}
R & v_s\\
0 & 1
\end{bmatrix},\qquad\\
\zeta &\triangleq
\begin{bmatrix}
\widehat\Omega & A_m\\
0 & 0
\end{bmatrix},\qquad\\
\gamma_v &\triangleq\;
\begin{bmatrix}
0_{3\times 1}\\
1
\end{bmatrix},
\end{align*}
the dynamics/output compactly read
\begin{align*}
\dot X &= X\,\zeta, \\
y_v &= X\,\gamma_v.
\end{align*}

\begin{align*}
\dot o_s &= v_s= X\,\gamma_v, \\
y_o &= o_s.
\end{align*}

---

Under a zero-order hold on $\zeta_k$ over the sampling interval,
\begin{align*}
X_{k+1}
&=
X_k\exp(\Delta t\,\zeta_k).
\end{align*}
For
$$
\zeta_k=
\begin{bmatrix}
\widehat\Omega_k&A_k^m\\
0&0
\end{bmatrix},
$$
the matrix exponential is
\begin{align*}
\exp(\Delta t\,\zeta_k)
&=
\begin{bmatrix}
\exp(\Delta t\,\widehat\Omega_k)
&
\Delta t\,J_l(\Delta t\Omega_k)A_k^m\\
0&1
\end{bmatrix},
\end{align*}
where
\begin{align*}
J_l(\Delta t\Omega_k)
&=
\Phi(-\Delta t\Omega_k)^{-1}.
\end{align*}
Therefore,
\begin{align*}
R_{k+1}
&=
R_k\exp(\Delta t\,\widehat\Omega_k),\\
v_{s,k+1}
&=
v_{s,k}
+
\Delta t\,R_kJ_l(\Delta t\Omega_k)A_k^m.
\end{align*}
Since
$$
J_l(\Delta t\Omega_k)=I+O(\Delta t),
$$
the first-order approximation is
\begin{align*}
v_{s,k+1}
&=
v_{s,k}+\Delta t\,R_kA_k^m+O((\Delta t)^2),\\
o_{s,k+1}
&=
o_{s,k}+\Delta t\,v_{s,k}+O((\Delta t)^2).
\end{align*}

---


The attitude, shifted velocity, and shifted position are denoted by
$$
R_k\in SO(3),\qquad
v_{s,k}\in\mathbb R^3,\qquad
o_{s,k}\in\mathbb R^3.
$$


Define the right-invariant attitude error by
$$
E_{R,k}
\triangleq
R_k\widetilde R_k^\top
=
\exp(\widehat{\eta_{R,k}}),
$$
and define the corresponding invariant velocity and position errors,
to first order, by
\begin{align*}
\eta_{v,k}
&\simeq
v_{s,k}-E_{R,k}\widetilde v_{s,k},\\
\eta_{o,k}
&\simeq
o_{s,k}-E_{R,k}\widetilde o_{s,k}.
\end{align*}
The local error state is therefore
$$
\eta_k
=
\begin{bmatrix}
\eta_{R,k}\\
\eta_{v,k}\\
\eta_{o,k}
\end{bmatrix}
\in\mathbb R^9.
$$

The linearized state-transition matrix is
$$
A_{k-1}
=
\begin{bmatrix}
I_{3\times3} & 0 & 0\\
0 & I_{3\times3} & 0\\
0 & \Delta t\,I_{3\times3} & I_{3\times3}
\end{bmatrix}.
$$

For GNSS measurements
$$
y_k=
\begin{bmatrix}
o_{s,k}\\
v_{s,k}
\end{bmatrix}
+z_k,
$$
the corresponding measurement Jacobian is
$$
H_k
=
\begin{bmatrix}
-\widehat{\widetilde o_{s,k}^-} & 0 & I_{3\times3}\\
-\widehat{\widetilde v_{s,k}^-} & I_{3\times3} & 0
\end{bmatrix}.
$$

We then also have:

$$
G_{k-1}^{X}
=
-\Delta t\,
\operatorname{Ad}_{\widetilde X_{k-1}}
\Phi(-\Delta t\,\zeta_{k-1})^{-1},
$$

$$
\operatorname{Ad}_{\widetilde X_{k-1}}
=
\begin{bmatrix}
\widetilde R_{k-1} & 0\\
\widehat{\widetilde v}_{s,k-1}\widetilde R_{k-1} & \widetilde R_{k-1}
\end{bmatrix}.
$$

$$
G_{k-1}
=
\begin{bmatrix}
G_{k-1}^{X}\\
0_{3\times6}
\end{bmatrix}
=
\begin{bmatrix}
-\Delta t\,
\operatorname{Ad}_{\widetilde X_{k-1}}
\Phi(-\Delta t\,\zeta_{k-1})^{-1}\\
0_{3\times6}
\end{bmatrix}.
$$

---

For
$$
\zeta_{k-1}
=
\begin{bmatrix}
\Omega_{k-1}\\
A^m_{k-1}
\end{bmatrix},
$$
we have
$$
\operatorname{ad}_{\zeta_{k-1}}
=
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat{A^m}_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}.
$$

Therefore,
\begin{align*}
\Phi(-\Delta t\,\zeta_{k-1})^{-1}
&=
I_{6\times6}
+
\frac{\Delta t}{2}\operatorname{ad}_{\zeta_{k-1}}
+
\frac{(\Delta t)^2}{6}\operatorname{ad}_{\zeta_{k-1}}^2
+
O((\Delta t)^3).
\end{align*}
Equivalently,
\begin{align*}
\Phi(-\Delta t\,\zeta_{k-1})^{-1}
&=
I_{6\times6}
+
\frac{\Delta t}{2}
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat{A^m}_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}\\
&\quad+
\frac{(\Delta t)^2}{6}
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat{A^m}_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}^{2}
+
O((\Delta t)^3).
\end{align*}

#### With Bias

We now consider a more complete IMU--GNSS sensor-fusion model including
gyroscope and accelerometer biases. Let
$$
R(t)\in SO(3),\qquad
v(t)\in\mathbb{R}^3,\qquad
o(t)\in\mathbb{R}^3
$$
denote the attitude, velocity, and position of the rigid body with respect
to an inertial frame. Let
$$
e_3=(0,0,1)^\top,
$$
and let $g>0$ denote the gravitational acceleration magnitude.

The IMU provides gyroscope and accelerometer measurements
\begin{align*}
\Omega^m
&=
\Omega+b_\Omega+n_\Omega,\\
A^m
&=
A+b_A+n_A,
\end{align*}
where
$$
b_\Omega,b_A\in\mathbb{R}^3
$$
are the gyroscope and accelerometer biases, and
\begin{align*}
n_\Omega&\sim\mathscr N(0,\Sigma_\Omega),\\
n_A&\sim\mathscr N(0,\Sigma_A)
\end{align*}
represent zero-mean IMU measurement noise.

Thus,
\begin{align*}
\Omega
&=
\Omega^m-b_\Omega-n_\Omega,\\
A
&=
A^m-b_A-n_A.
\end{align*}

The physical continuous-time inertial-navigation model is
\begin{align*}
\dot R
&=
R\widehat{\Omega},\\
\dot v
&=
RA-ge_3,\\
\dot o
&=
v.
\end{align*}

Introduce the shifted velocity and position
\begin{align*}
v_s(t)
&\triangleq
v(t)+gt\,e_3,\\
o_s(t)
&\triangleq
o(t)+\frac{1}{2}gt^2e_3.
\end{align*}
Then
\begin{align*}
\dot v_s
&=
\dot v+ge_3
=
RA,\\
\dot o_s
&=
v_s.
\end{align*}

Hence the shifted continuous-time navigation model is
\begin{align*}
\dot R
&=
R\widehat{\Omega},\\
\dot v_s
&=
RA,\\
\dot o_s
&=
v_s.
\end{align*}

Expressed in terms of the measured IMU signals and biases, this becomes
\begin{align*}
\dot R
&=
R\widehat{\left(
\Omega^m-b_\Omega-n_\Omega
\right)},\\
\dot v_s
&=
R\left(
A^m-b_A-n_A
\right),\\
\dot o_s
&=
v_s.
\end{align*}

The IMU biases are modeled as random walks. Formally,
\begin{align*}
\dot b_\Omega
&=
w_\Omega,\\
\dot b_A
&=
w_A,
\end{align*}
where $w_\Omega$ and $w_A$ are zero-mean white-noise processes with
spectral-density matrices $\Sigma_{b_\Omega}$ and $\Sigma_{b_A}$,
respectively.

For a sampling period $\Delta t$, the corresponding discrete bias model is
\begin{align*}
b_{\Omega,k+1}
&=
b_{\Omega,k}+\nu_{\Omega,k},\\
b_{A,k+1}
&=
b_{A,k}+\nu_{A,k},
\end{align*}
where
\begin{align*}
\nu_{\Omega,k}
&\sim
\mathscr N\left(
0,\Delta t\,\Sigma_{b_\Omega}
\right),\\
\nu_{A,k}
&\sim
\mathscr N\left(
0,\Delta t\,\Sigma_{b_A}
\right).
\end{align*}

For the nominal filter propagation, define the bias-corrected IMU inputs
\begin{align*}
\widetilde\omega_k
&\triangleq
\Omega_k^m-\widetilde b_{\Omega,k},\\
\widetilde a_k
&\triangleq
A_k^m-\widetilde b_{A,k}.
\end{align*}

Under a zero-order hold on the corrected IMU inputs, the nominal attitude
propagation is
\begin{align*}
\widetilde R_{k+1}^-
&=
\widetilde R_k
\exp\left(
\Delta t\,\widehat{\widetilde\omega}_k
\right).
\end{align*}

The corresponding shifted-velocity propagation is
\begin{align*}
\widetilde v_{s,k+1}^-
&=
\widetilde v_{s,k}
+
\Delta t\,
\widetilde R_k
J_l(\Delta t\,\widetilde\omega_k)
\widetilde a_k,
\end{align*}
where
\begin{align*}
J_l(\Delta t\,\widetilde\omega_k)
&=
\Phi(-\Delta t\,\widetilde\omega_k)^{-1}.
\end{align*}

Since
$$
J_l(\Delta t\,\widetilde\omega_k)
=
I+O(\Delta t),
$$
the usual first-order propagation is
\begin{align*}
\widetilde v_{s,k+1}^-
&=
\widetilde v_{s,k}
+
\Delta t\,
\widetilde R_k\widetilde a_k
+
O((\Delta t)^2).
\end{align*}

Using the corresponding second-order position approximation gives
\begin{align*}
\widetilde o_{s,k+1}^-
&=
\widetilde o_{s,k}
+
\Delta t\,\widetilde v_{s,k}
+
\frac{1}{2}(\Delta t)^2
\widetilde R_k\widetilde a_k
+
O((\Delta t)^3).
\end{align*}

The nominal bias estimates are propagated as
\begin{align*}
\widetilde b_{\Omega,k+1}^-
&=
\widetilde b_{\Omega,k},\\
\widetilde b_{A,k+1}^-
&=
\widetilde b_{A,k},
\end{align*}
while the random-walk uncertainty is incorporated through the process
covariance.

The complete shifted navigation state is therefore
$$
\mathscr X_k
=
\left(
R_k,\,
v_{s,k},\,
o_{s,k},\,
b_{\Omega,k},\,
b_{A,k}
\right).
$$

GNSS measurements are naturally obtained in the physical variables
$(o_k,v_k)$. For use with the shifted state, define the corresponding
shifted GNSS measurements by
\begin{align*}
y_{o_s,k}^{\mathrm{GNSS}}
&\triangleq
y_{o,k}^{\mathrm{GNSS}}
+
\frac{1}{2}gt_k^2e_3,\\
y_{v_s,k}^{\mathrm{GNSS}}
&\triangleq
y_{v,k}^{\mathrm{GNSS}}
+
gt_ke_3.
\end{align*}

Thus, when GNSS provides both position and velocity,
\begin{align*}
y_{s,k}^{\mathrm{GNSS}}
&=
\begin{bmatrix}
o_{s,k}\\
v_{s,k}
\end{bmatrix}
+
z_k,
\qquad
z_k\sim
\mathscr N(0,\Sigma_{\mathrm{GNSS}}).
\end{align*}

For position-only GNSS,
\begin{align*}
y_{o_s,k}^{\mathrm{GNSS}}
&=
o_{s,k}+z_k,
\qquad
z_k\sim\mathscr N(0,\Sigma_o).
\end{align*}

Thus, using the usual first-/second-order inertial-navigation
discretization, the nominal propagation in the shifted variables is
\begin{align*}
\widetilde R_{k+1}^-
&=
\widetilde R_k
\exp\left(
\Delta t\,
\widehat{\Omega_k^m-\widetilde b_{\Omega,k}}
\right),\\
\widetilde v_{s,k+1}^-
&=
\widetilde v_{s,k}
+
\Delta t\,
\widetilde R_k
\left(
A_k^m-\widetilde b_{A,k}
\right)
+
O((\Delta t)^2),\\
\widetilde o_{s,k+1}^-
&=
\widetilde o_{s,k}
+
\Delta t\,\widetilde v_{s,k}
+
\frac{1}{2}(\Delta t)^2
\widetilde R_k
\left(
A_k^m-\widetilde b_{A,k}
\right)
+
O((\Delta t)^3),\\
\widetilde b_{\Omega,k+1}^-
&=
\widetilde b_{\Omega,k},\\
\widetilde b_{A,k+1}^-
&=
\widetilde b_{A,k}.
\end{align*}

**Note that:** Since the shifted variables $(v_s,o_s)$ are used,
the explicit gravity terms disappear from the propagation equations.
Consequently, the gravity coupling
$$
-g\Delta t\,\widehat e_3
$$
also disappears from the corresponding invariant error-state transition
matrix, recovering the particularly clean invariant $A_{k-1}$ structure
of the preceding formulation.

---

Define the augmented local error state
$$
\eta_k=
\begin{bmatrix}
\eta_{R,k}&
\eta_{v_s,k}&
\eta_{o_s,k}&
\eta_{b_\Omega,k}&
\eta_{b_A,k}
\end{bmatrix}^{\top}
\in\mathbb R^{15},
$$
with right-invariant attitude, shifted-velocity, and shifted-position
errors, together with additive gyro- and accelerometer-bias errors.

Let
$$
\widetilde\zeta_{k-1}
=
\begin{bmatrix}
\widetilde\omega_{k-1}\\
\widetilde a_{k-1}
\end{bmatrix},
$$
and define
$$
G_{k-1}^{X}
=
-\Delta t\,
\operatorname{Ad}_{\widetilde X_{k-1}}
\Phi(-\Delta t\,\widetilde\zeta_{k-1})^{-1}
=
\begin{bmatrix}
G_{\Omega,k-1}&G_{A,k-1}
\end{bmatrix},
$$
where
$$
G_{\Omega,k-1},G_{A,k-1}\in\mathbb R^{6\times3}.
$$

Writing
$$
G_{\Omega,k-1}
=
\begin{bmatrix}
G_{\Omega,k-1}^{R}\\
G_{\Omega,k-1}^{v_s}
\end{bmatrix},
\qquad
G_{A,k-1}
=
\begin{bmatrix}
G_{A,k-1}^{R}\\
G_{A,k-1}^{v_s}
\end{bmatrix},
$$
the first-order augmented state-transition matrix is
$$
A_{k-1}
=
\begin{bmatrix}
I_3 & 0 & 0
    & G_{\Omega,k-1}^{R}
    & G_{A,k-1}^{R}\\
0 & I_3 & 0
    & G_{\Omega,k-1}^{v_s}
    & G_{A,k-1}^{v_s}\\
0 & \Delta t\,I_3 & I_3
    & 0 & 0\\
0 & 0 & 0
    & I_3 & 0\\
0 & 0 & 0
    & 0 & I_3
\end{bmatrix}.
$$

For process noise ordered as
$$
\begin{bmatrix}
n_\Omega^\top&
n_A^\top&
\nu_{b_\Omega}^\top&
\nu_{b_A}^\top
\end{bmatrix}^{\top},
$$
the corresponding process-noise input matrix is
$$
G_{k-1}
=
\begin{bmatrix}
G_{k-1}^{X} & 0_{6\times6}\\
0_{3\times6} & 0_{3\times6}\\
0_{6\times6} & I_6
\end{bmatrix}.
$$

For shifted GNSS position and velocity measurements,
$$
y_{s,k}^{\mathrm{GNSS}}
=
\begin{bmatrix}
o_{s,k}\\
v_{s,k}
\end{bmatrix}
+z_k,
$$
the measurement matrix is
$$
H_k
=
\begin{bmatrix}
-\widehat{\widetilde o_{s,k}^{-}}
    & 0 & I_3 & 0 & 0\\
-\widehat{\widetilde v_{s,k}^{-}}
    & I_3 & 0 & 0 & 0
\end{bmatrix}.
$$

**Note that:** Since the shifted variables $(v_s,o_s)$ are used,
the gravity-induced coupling
$$
-g\Delta t\,\widehat e_3
$$
that appears in the unshifted formulation disappears from
$A_{k-1}$. The $(R,v_s)$ subsystem therefore recovers the particularly
clean invariant error structure, while the only additional deterministic
coupling is the kinematic relation from shifted velocity to shifted
position.

# Continuous-Time Intrinsic EKF on Lie Groups

Let $G$ be an $n$-dimensional Lie group with Lie algebra $\mathfrak g$,
and let
$$
(g,\zeta)\in G\times\mathfrak g.
$$
Let
$$
\phi:G\times M\to M
$$
be a left linear action on an $m$-dimensional vector space $M$, so that,
for each $g\in G$,
$$
\phi_g\in\mathrm{GL}(M).
$$
We recall the identities
\begin{align*}
g\exp(\zeta)g^{-1}
&=
\exp\left(
\operatorname{Ad}_g\zeta
\right),\\
\phi_{\exp(\zeta)}
&=
\exp\left(
T_e\phi(\zeta)
\right).
\end{align*}
In particular, taking $M=\mathfrak g$ and $\phi=\operatorname{Ad}$ gives
\begin{align*}
\operatorname{Ad}_{\exp(\zeta)}
&=
\exp(\operatorname{ad}_\zeta).
\end{align*}

We consider the continuous-time kinematic model
\begin{align*}
\dot g
&=
g\big(\zeta+n_\zeta\big),\\
y
&=
\phi_{g^{-1}}(\gamma)+n,
\end{align*}
where $\zeta(t)\in\mathfrak g$ and $\gamma\in M$ are known, while
$n_\zeta$ and $n$ represent zero-mean process and measurement noise
with covariance intensities $\Sigma_\zeta$ and $\Sigma_y$, respectively.

For notational simplicity, the following uses the conventional formal
continuous-time white-noise notation. A rigorous stochastic formulation
may equivalently be written as the corresponding stochastic differential
equations, with a specified It\^o or Stratonovich interpretation.

Consider the intrinsic observer
\begin{align*}
\dot{\widetilde g}
&=
\widetilde g
\left(
\zeta+K(t)(y-\widetilde y)
\right),\\
\widetilde y
&=
\phi_{\widetilde g^{-1}}(\gamma).
\end{align*}

Define the left-invariant estimation error
\begin{align*}
e_g
&\triangleq
\widetilde g^{-1}g
=
\exp(\eta_e),
\qquad
\eta_e\in\mathfrak g,
\end{align*}
and the output error
\begin{align*}
y_e
&\triangleq
y-\widetilde y.
\end{align*}

Since
$$
g=\widetilde g e_g,
$$
we have
\begin{align*}
g^{-1}
&=
e_g^{-1}\widetilde g^{-1},
\end{align*}
and therefore
\begin{align*}
y_e
&=
\left(
\phi_{e_g^{-1}}-I
\right)
\widetilde y+n.
\end{align*}

The exact group-error dynamics are
\begin{align*}
\dot e_g
&=
e_g
\left[
\left(
I-\operatorname{Ad}_{e_g^{-1}}
\right)\zeta
-
\operatorname{Ad}_{e_g^{-1}}K(t)y_e
+
n_\zeta
\right].
\end{align*}

Using
$$
e_g=\exp(\eta_e),
$$
the corresponding exact log-error dynamics are
\begin{align*}
\dot\eta_e
&=
-\operatorname{ad}_{\zeta}\eta_e
-
\left(
\frac{\operatorname{ad}_{\eta_e}}
{\exp(\operatorname{ad}_{\eta_e})-I}
\right)
K(t)y_e\\
&\quad+
\left(
\frac{
\exp(\operatorname{ad}_{\eta_e})
\operatorname{ad}_{\eta_e}}
{\exp(\operatorname{ad}_{\eta_e})-I}
\right)
n_\zeta.
\end{align*}

The measurement linearization is obtained from
\begin{align*}
y_e
&=
\left(
\phi_{\exp(-\eta_e)}-I
\right)
\widetilde y+n.
\end{align*}
Define the linear operator
\begin{align*}
H(t)\xi
&\triangleq
-
\left(
T_e\phi(\xi)
\right)\widetilde y,
\qquad
\xi\in\mathfrak g.
\end{align*}
Equivalently,
\begin{align*}
H(t)\xi
&=
-\phi_{\widetilde g^{-1}}
\left(
\left(
T_e\phi\circ
\operatorname{Ad}_{\widetilde g}
\right)\xi(\gamma)
\right).
\end{align*}
Then
\begin{align*}
y_e
&=
H(t)\eta_e+n+O(\|\eta_e\|^2).
\end{align*}

Define
\begin{align*}
A(t)
&\triangleq
-\operatorname{ad}_{\zeta(t)}.
\end{align*}
Neglecting higher-order terms in the local error gives the linearized
intrinsic error model
\begin{align*}
\dot\eta_e
&=
\left(
A(t)-K(t)H(t)
\right)\eta_e
+
n_\zeta
-
K(t)n.
\end{align*}

The corresponding Kalman--Bucy covariance and gain equations are
\begin{align*}
\dot P
&=
A P+P A^\top
-
P H^\top\Sigma_y^{-1}H P
+
\Sigma_\zeta,\\
K
&=
P H^\top\Sigma_y^{-1}.
\end{align*}

Equivalently, substituting the Kalman gain into the linearized
closed-loop error dynamics gives
\begin{align*}
\dot P
&=
(A-KH)P
+
P(A-KH)^\top
+
\Sigma_\zeta
+
K\Sigma_yK^\top.
\end{align*}

If the mean error is denoted by
$$
m_{\eta_e}(t)
\triangleq
\mathbb E[\eta_e(t)],
$$
then, to first order,
\begin{align*}
\dot m_{\eta_e}
&=
(A-KH)m_{\eta_e}.
\end{align*}
The error covariance is defined by
\begin{align*}
P(t)
&\triangleq
\mathbb E
\left[
(\eta_e-m_{\eta_e})
(\eta_e-m_{\eta_e})^\top
\right].
\end{align*}

Under the standard observability, stabilizability, and boundedness
conditions for the linearized system, the deterministic local error
dynamics are asymptotically stable and the stochastic local error
remains bounded with covariance governed, to first order, by the
Riccati equation above. In the presence of persistent process and
measurement noise, pointwise convergence of the estimation error to
zero is not generally expected. For a time-invariant detectable and
stabilizable linearization, the Riccati equation may additionally
converge to its steady-state solution.

## Comparison with the Discrete Intrinsic EKF

There is an important distinction between the continuous-time
formulation above and the discrete intrinsic EKF developed subsequently.
The continuous-time derivation above uses the left-invariant error
$$
e_g^{L}
=
\widetilde g^{-1}g,
$$
for which the common kinematic input does not cancel. Its first-order
deterministic error dynamics are
\begin{align*}
\dot\eta_e^{L}
&=
-\operatorname{ad}_{\zeta}\eta_e^{L},
\end{align*}
so that
$$
A_c=-\operatorname{ad}_{\zeta}.
$$

In the discrete formulation, however, we use the right-invariant error
$$
e_k^{R}
=
g_k\widetilde g_k^{-1}.
$$
For the deterministic kinematics
\begin{align*}
g_k
&=
g_{k-1}\exp(\Delta T\zeta_{k-1}),\\
\widetilde g_k^-
&=
\widetilde g_{k-1}
\exp(\Delta T\zeta_{k-1}),
\end{align*}
the common group displacement cancels from the right-invariant error.
Consequently,
\begin{align*}
\eta_k^-
&=
\eta_{k-1}
+
G_{k-1}w_{k-1}
+
H.O.T.,
\end{align*}
and hence
$$
A_d=I.
$$

This distinction is caused by the choice of invariant error, rather
than by discretization itself. Indeed, if the right-invariant error
$$
e_g^{R}=g\widetilde g^{-1}
$$
is used for the corresponding noise-free continuous-time kinematics,
then
\begin{align*}
\dot e_g^{R}
&=
0,
\end{align*}
and therefore
\begin{align*}
\dot\eta_e^{R}
&=
0.
\end{align*}
Thus the continuous-time error-transition matrix is
$$
A_c^{R}=0,
$$
whose discrete-time transition is
$$
\exp(A_c^{R}\Delta T)=I.
$$

Hence the discrete result
$$
A_d=I
$$
is exactly consistent with the continuous-time right-invariant error
dynamics. This is the Lie-group analogue of ordinary kinematics
$\dot x=v$: when two trajectories are driven by the same velocity,
their appropriately defined relative error remains unchanged.

# Discrete-Time Pre-Observers on Lie Groups with Time-Invariant Error Dynamics

The purpose of this section is to introduce discrete-time pre-observers
whose intrinsic estimation-error dynamics are independent of the true
trajectory. Depending on the choice of group action and invariant error,
the resulting error dynamics may additionally be independent of the
known input and hence autonomous.

Let $G$ be an $n$-dimensional Lie group with Lie algebra $\mathfrak g$,
and let
$$
(g,\zeta)\in G\times\mathfrak g.
$$
Let $M$ be an $m$-dimensional manifold and let
$$
\phi:G\times M\to M
$$
denote either a left or a right group action. We consider the
kinematic system
\begin{align*}
\dot g
&=
g\zeta,\\
y
&=
\phi_g(\gamma),
\end{align*}
where $\zeta(t)\in\mathfrak g$ is a known input and $\gamma\in M$ is a
known constant.

Under a zero-order hold on $\zeta$ over the sampling interval,
\begin{align*}
g_k
&=
g_{k-1}\exp(\Delta T\zeta_{k-1}).
\end{align*}
Define
\begin{align*}
u_{k-1}
&\triangleq
\exp(\Delta T\zeta_{k-1}).
\end{align*}

The innovation map is taken as
$$
L:M\times M\to\mathfrak g,
$$
with
$$
L(y,y)=0.
$$
We assume that $L$ is invariant under the simultaneous action of $G$:
\begin{align*}
L\left(
\phi_h(y_1),\phi_h(y_2)
\right)
&=
L(y_1,y_2),
\qquad h\in G.
\end{align*}




## Outputs Induced by a Left Action

Suppose $\phi$ is a left action, so that
\begin{align*}
\phi_g\circ\phi_h
&=
\phi_{gh}.
\end{align*}

Consider the pre-observer
\begin{align*}
\widetilde g_k^-
&=
\widetilde g_{k-1}u_{k-1},\\
\widetilde y_k^-
&=
\phi_{\widetilde g_k^-}(\gamma),\\
\widetilde g_k
&=
\widetilde g_k^-
\exp\left(
L(y_k,\widetilde y_k^-)
\right).
\end{align*}

Define the left-invariant a priori and a posteriori errors by
\begin{align*}
e_k^-
&\triangleq
(\widetilde g_k^-)^{-1}g_k,\\
e_k
&\triangleq
\widetilde g_k^{-1}g_k.
\end{align*}

The prediction error satisfies
\begin{align*}
e_k^-
&=
(\widetilde g_{k-1}u_{k-1})^{-1}
g_{k-1}u_{k-1}\\
&=
u_{k-1}^{-1}
\widetilde g_{k-1}^{-1}
g_{k-1}
u_{k-1}\\
&=
u_{k-1}^{-1}e_{k-1}u_{k-1}.
\end{align*}

After the correction,
\begin{align*}
e_k
&=
\widetilde g_k^{-1}g_k\\
&=
\exp\left(
-L(y_k,\widetilde y_k^-)
\right)e_k^-.
\end{align*}

Using the invariance of $L$ and the left-action property,
\begin{align*}
L(y_k,\widetilde y_k^-)
&=
L\left(
\phi_{g_k}(\gamma),
\phi_{\widetilde g_k^-}(\gamma)
\right)\\
&=
L\left(
\phi_{(\widetilde g_k^-)^{-1}}
\phi_{g_k}(\gamma),
\phi_{(\widetilde g_k^-)^{-1}}
\phi_{\widetilde g_k^-}(\gamma)
\right)\\
&=
L\left(
\phi_{(\widetilde g_k^-)^{-1}g_k}(\gamma),
\gamma
\right)\\
&=
L\left(
\phi_{e_k^-}(\gamma),
\gamma
\right).
\end{align*}

Therefore, the complete intrinsic error recursion is
\begin{align*}
e_k^-
&=
u_{k-1}^{-1}e_{k-1}u_{k-1},\\
e_k
&=
\exp\left(
-L\left(\phi_{e_k^-}(\gamma),\gamma\right)
\right)e_k^-.
\end{align*}

Thus the error dynamics are independent of the true trajectory $g_k$.
They generally retain dependence on the known input $u_{k-1}$ through
the conjugation
$$
u_{k-1}^{-1}e_{k-1}u_{k-1}.
$$
Hence, for time-varying $\zeta$, these dynamics are
trajectory-independent but not strictly autonomous.



## Outputs Induced by a Right Action

Suppose instead that $\phi$ is a right action, so that
\begin{align*}
\phi_g\circ\phi_h
&=
\phi_{hg}.
\end{align*}

Consider the pre-observer
\begin{align*}
\widetilde g_k^-
&=
\widetilde g_{k-1}u_{k-1},\\
\widetilde y_k^-
&=
\phi_{\widetilde g_k^-}(\gamma),\\
\widetilde g_k
&=
\exp\left(
L(y_k,\widetilde y_k^-)
\right)
\widetilde g_k^-.
\end{align*}

Define the right-invariant a priori and a posteriori errors by
\begin{align*}
e_k^-
&\triangleq
g_k(\widetilde g_k^-)^{-1},\\
e_k
&\triangleq
g_k\widetilde g_k^{-1}.
\end{align*}

The prediction error satisfies
\begin{align*}
e_k^-
&=
g_{k-1}u_{k-1}
(\widetilde g_{k-1}u_{k-1})^{-1}\\
&=
g_{k-1}
\widetilde g_{k-1}^{-1}\\
&=
e_{k-1}.
\end{align*}

After the correction,
\begin{align*}
e_k
&=
g_k\widetilde g_k^{-1}\\
&=
e_k^-
\exp\left(
-L(y_k,\widetilde y_k^-)
\right).
\end{align*}

Using the invariance of $L$ and the right-action property,
\begin{align*}
L(y_k,\widetilde y_k^-)
&=
L\left(
\phi_{g_k}(\gamma),
\phi_{\widetilde g_k^-}(\gamma)
\right)\\
&=
L\left(
\phi_{(\widetilde g_k^-)^{-1}}
\phi_{g_k}(\gamma),
\phi_{(\widetilde g_k^-)^{-1}}
\phi_{\widetilde g_k^-}(\gamma)
\right)\\
&=
L\left(
\phi_{g_k(\widetilde g_k^-)^{-1}}(\gamma),
\gamma
\right)\\
&=
L\left(
\phi_{e_k^-}(\gamma),
\gamma
\right).
\end{align*}

Therefore, the complete intrinsic error recursion is
\begin{align*}
e_k^-
&=
e_{k-1},\\
e_k
&=
e_{k-1}
\exp\left(
-L\left(
\phi_{e_{k-1}}(\gamma),
\gamma
\right)
\right).
\end{align*}

In this case, the intrinsic error dynamics depend neither on the true
trajectory $g_k$ nor on the known input $\zeta_k$. They are therefore
autonomous.

In particular, the prediction step satisfies
$$
e_k^-=e_{k-1},
$$
which is the nonlinear Lie-group analogue of the Euclidean kinematic
property that two trajectories driven by the same velocity retain the
same relative error. Linearizing around the identity therefore gives
$$
A_{k-1}=I,
$$
which is precisely the structure exploited in the discrete intrinsic
EKF developed subsequently.

# References

[1] K. C. Wolfe, M. Mashner, and G. S. Chirikjian, “Bayesian fusion on Lie groups,” *Journal of Algebraic Statistics*, 2(1):75–97, 2011. [Link](https://rpk.lcsr.jhu.edu/publications/)

[2] S. Bonnable, P. Martin, and E. Salan, “Invariant extended Kalman filter: theory and application to a velocity-aided attitude estimation problem,” *Proc. 48th IEEE CDC/28th CCC*, pp. 1297–1304, Dec 2009. [Link](https://doi.org/10.1109/CDC.2009.5399990)

[3] S. Bonnabel, “Left-invariant extended Kalman filter and attitude estimation,” *Proc. 46th IEEE CDC*, pp. 1027–1032, Dec 2007. [Link](https://doi.org/10.1109/CDC.2007.4434662)

[4] G. Bourmaud, R. Mégret, A. Giremus, and Y. Berthoumieu, “Discrete extended Kalman filter on Lie groups,” *EUSIPCO 2013*, pp. 1–5, Sept 2013. [Link](https://hal.science/hal-00903252/document)

[5] G. S. Chirikjian, “Information theory on Lie groups and mobile robotics applications,” *Proc. 2010 IEEE ICRA*, pp. 2751–2757, May 2010. [Link](https://rpk.lcsr.jhu.edu/publications/)

[6] Y. Wang and G. S. Chirikjian, “Error propagation on the Euclidean group with applications to manipulator kinematics,” *IEEE Trans. Robotics*, 22(4):591–602, Aug 2006. [Link](https://ieeexplore.ieee.org/document/1673946)

[7] M. J. Piggott and V. Solo, “Stochastic numerical analysis for Brownian motion on SO(3),” *Proc. 53rd IEEE CDC*, pp. 3420–3425, Dec 2014. [Link](https://doi.org/10.1109/CDC.2014.7039919)

[8] O. Tuzel, F. Porikli, and P. Meer, “Learning on Lie groups for invariant detection and tracking,” *Proc. 2008 IEEE CVPR*, pp. 1–8, June 2008. [Link](https://doi.org/10.1109/CVPR.2008.4587521)

[9] A. Barrau and S. Bonnabel, “Intrinsic filtering on Lie groups with applications to attitude estimation,” *IEEE Trans. Automatic Control*, 60(2):436–449, Feb 2015. [Link](https://doi.org/10.1109/TAC.2014.2342911)

[10] S. Bonnabel and A. Barrau, “An intrinsic Cramér–Rao bound on SO(3) for (dynamic) attitude filtering,” *Proc. 54th IEEE CDC*, pp. 2158–2163, Dec 2015. [Link](https://dblp.org/rec/conf/cdc/BonnabelB15)

[11] A. Barrau and S. Bonnabel, “The invariant extended Kalman filter as a stable observer,” *IEEE Trans. Automatic Control*, 62(4):1797–1812, Apr 2017. [Link](https://doi.org/10.1109/TAC.2016.2594085)

[12] C. Lageman, J. Trumpf, and R. Mahony, “Gradient-like observers for invariant dynamics on a Lie group,” *IEEE Trans. Automatic Control*, 55(2):367–377, Feb 2010. [Link](https://doi.org/10.1109/TAC.2009.2034937)

[13] S. Bonnabel, P. Martin, and P. Rouchon, “Non-linear symmetry-preserving observers on Lie groups,” *IEEE Trans. Automatic Control*, 54(7):1709–1713, July 2009. [Link](https://doi.org/10.1109/TAC.2009.2020646)

[14] M. Izadi and A. K. Sanyal, “Rigid body attitude estimation based on the Lagrange–d’Alembert principle,” *Automatica*, 50(10):2570–2577, 2014. [Link](https://doi.org/10.1016/j.automatica.2014.08.010)

[15] S. Bonnabel and J. J. Slotine, “A contraction theory-based analysis of the stability of the deterministic extended Kalman filter,” *IEEE Trans. Automatic Control*, 60(2):565–569, Feb 2015. [Link](https://doi.org/10.1109/TAC.2014.2336991)
